wearable-sensor-dashboard/
├─ app.py
├─ wearable/
│  ├─ __init__.py
│  └─ core.py
├─ tests/
│  └─ test_core.py
├─ data/
│  └─ sample.csv
├─ requirements.txt
└─ .github/
   └─ workflows/
      └─ ci.yml






High-level design

Data model: timestamped sensor records (e.g., heart_rate, steps, temperature). Parse → validate → clean → compute metrics.

Backend (pure Python): deterministic functions for ingest, clean, resample, and KPIs. 100% covered by unit tests (TDD).

Frontend (Streamlit): file uploader → calls backend → charts (time series + daily aggregates) → KPI cards + downloadable CSV.

CI/CD: GitHub Actions runs tests on every push/PR; if green, Streamlit Community Cloud will auto-redeploy from the same repo (simple & reliable CD).
(If you prefer GH Actions to push to Hugging Face Spaces or another host, I included notes at the bottom.)

Backend

In [ ]:
from __future__ import annotations
import io
from dataclasses import dataclass
from typing import List, Optional, Tuple
import pandas as pd
import numpy as np

# ---- Data schema ----
REQUIRED_COLS = ["timestamp"]
ALLOWED_SENSORS = {"heart_rate", "steps", "temperature"}

@dataclass(frozen=True)
class Metrics:
    n_rows: int
    time_start: pd.Timestamp
    time_end: pd.Timestamp
    duration_hours: float
    sensors_present: List[str]
    daily_means: pd.DataFrame
    daily_max: pd.DataFrame
    resting_hr: Optional[float]
    step_total: Optional[int]
    temp_mean: Optional[float]

def _coerce_ts(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_convert(None)

def read_csv(file_or_bytes) -> pd.DataFrame:
    """Read CSV from path/bytes/IO and return raw DataFrame (no cleaning)."""
    if isinstance(file_or_bytes, (bytes, bytearray)):
        buf = io.BytesIO(file_or_bytes)
        df = pd.read_csv(buf)
    else:
        df = pd.read_csv(file_or_bytes)
    return df

def validate_and_clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Ensure required cols
    - Parse timestamps
    - Keep only known sensor cols
    - Drop rows with invalid ts or all-NaN sensor values
    - Remove duplicates; sort by time
    """
    missing = set(REQUIRED_COLS) - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.copy()
    df["timestamp"] = _coerce_ts(df["timestamp"])
    if df["timestamp"].isna().any():
        df = df.dropna(subset=["timestamp"])

    # Keep allowed sensor columns (present in df)
    sensor_cols = [c for c in df.columns if c in ALLOWED_SENSORS]
    if not sensor_cols:
        raise ValueError("No recognized sensor columns found.")

    # Coerce numeric
    for c in sensor_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop rows where all sensors are NaN
    df = df.dropna(subset=sensor_cols, how="all")

    # De-dup & sort
    df = df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp")
    return df[["timestamp"] + sensor_cols]

def resample_daily(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return (daily_means, daily_max) for all sensor columns."""
    df = df.set_index("timestamp")
    rule = "1D"
    agg_mean = df.resample(rule).mean(numeric_only=True)
    agg_max = df.resample(rule).max(numeric_only=True)
    # Keep only columns that exist
    return agg_mean, agg_max

def compute_metrics(df: pd.DataFrame) -> Metrics:
    sensor_cols = [c for c in df.columns if c in ALLOWED_SENSORS]
    daily_means, daily_max = resample_daily(df)

    # Simple examples of domain KPIs:
    resting_hr = None
    if "heart_rate" in sensor_cols:
        # Resting HR proxy: 5th percentile of HR across the period
        resting_hr = float(np.nanpercentile(df["heart_rate"], 5))

    step_total = None
    if "steps" in sensor_cols:
        # steps column often comes as per-interval counts; sum them
        step_total = int(np.nansum(df["steps"]))

    temp_mean = None
    if "temperature" in sensor_cols:
        temp_mean = float(np.nanmean(df["temperature"]))

    time_start = df["timestamp"].min()
    time_end   = df["timestamp"].max()
    duration_hours = (time_end - time_start).total_seconds() / 3600.0

    return Metrics(
        n_rows=len(df),
        time_start=time_start,
        time_end=time_end,
        duration_hours=duration_hours,
        sensors_present=sensor_cols,
        daily_means=daily_means,
        daily_max=daily_max,
        resting_hr=resting_hr,
        step_total=step_total,
        temp_mean=temp_mean,
    )



TDD TEST

In [ ]:
import io
import pandas as pd
import numpy as np
import pytest
from wearable.core import read_csv, validate_and_clean, compute_metrics

def make_csv(text: str) -> bytes:
    return text.encode("utf-8")

def test_read_and_clean_basic():
    csv = make_csv(
        "timestamp,heart_rate,steps,temperature,ignore\n"
        "2025-01-01T00:00:00Z,70,0,36.8,x\n"
        "2025-01-01T00:01:00Z,72,2,36.9,x\n"
    )
    raw = read_csv(io.BytesIO(csv))
    clean = validate_and_clean(raw)
    assert "timestamp" in clean.columns
    assert "heart_rate" in clean.columns
    assert "steps" in clean.columns
    assert "temperature" in clean.columns
    assert "ignore" not in clean.columns
    assert len(clean) == 2

def test_drop_invalid_timestamps():
    csv = make_csv(
        "timestamp,heart_rate\n"
        "badtime,70\n"
        "2025-01-01T00:00:00Z,71\n"
    )
    raw = read_csv(io.BytesIO(csv))
    clean = validate_and_clean(raw)
    assert len(clean) == 1
    assert pd.Timestamp("2025-01-01T00:00:00") == clean["timestamp"].iloc[0]

def test_all_nan_sensor_row_dropped():
    csv = make_csv(
        "timestamp,heart_rate,steps\n"
        "2025-01-01T00:00:00Z,,\n"
        "2025-01-01T00:01:00Z,70,1\n"
    )
    raw = read_csv(io.BytesIO(csv))
    clean = validate_and_clean(raw)
    assert len(clean) == 1
    assert clean["steps"].iloc[0] == 1

def test_compute_metrics():
    csv = make_csv(
        "timestamp,heart_rate,steps,temperature\n"
        "2025-01-01T00:00:00Z,70,0,36.7\n"
        "2025-01-01T00:10:00Z,60,25,36.9\n"
        "2025-01-02T01:00:00Z,65,80,37.0\n"
    )
    raw = read_csv(io.BytesIO(csv))
    clean = validate_and_clean(raw)
    m = compute_metrics(clean)
    assert m.n_rows == 3
    assert "heart_rate" in m.sensors_present
    assert m.step_total == 105
    assert m.resting_hr is not None
    assert m.daily_means.shape[0] >= 2  # two days


STREAMLIT UI

In [ ]:
import pandas as pd
import streamlit as st
from wearable.core import read_csv, validate_and_clean, compute_metrics

st.set_page_config(page_title="Wearable Sensor Data Analyzer", layout="wide")
st.title("⌚ Wearable Sensor Data Analyzer")

with st.sidebar:
    st.markdown("### Data Upload")
    uploaded = st.file_uploader("Upload CSV", type=["csv"])
    st.markdown("---")
    st.caption("CSV must include `timestamp` and any of: `heart_rate`, `steps`, `temperature`.")

if uploaded is None:
    st.info("Upload a CSV to begin. A small sample is shown below.")
    sample = pd.read_csv("data/sample.csv")
    st.dataframe(sample.head(12), use_container_width=True)
else:
    try:
        raw_df = read_csv(uploaded)
        clean_df = validate_and_clean(raw_df)
        metrics = compute_metrics(clean_df)

        st.success("Data loaded and validated.")
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Rows", f"{metrics.n_rows}")
        c2.metric("Duration (hrs)", f"{metrics.duration_hours:.1f}")
        c3.metric("Sensors", ", ".join(metrics.sensors_present) or "—")
        c4.metric("Total Steps", f"{metrics.step_total:,}" if metrics.step_total is not None else "—")

        c5, c6, c7 = st.columns(3)
        if metrics.resting_hr is not None:
            c5.metric("Resting HR (≈5th %tile)", f"{metrics.resting_hr:.0f} bpm")
        if metrics.temp_mean is not None:
            c6.metric("Avg Temperature", f"{metrics.temp_mean:.1f} °C")
        c7.metric("Start → End", f"{metrics.time_start} → {metrics.time_end}")

        st.subheader("Time Series")
        st.line_chart(clean_df.set_index("timestamp"))

        st.subheader("Daily Averages")
        st.bar_chart(metrics.daily_means)

        with st.expander("Data (clean)"):
            st.dataframe(clean_df, use_container_width=True)

        # Downloadables
        st.download_button(
            "Download Cleaned CSV",
            data=clean_df.to_csv(index=False).encode("utf-8"),
            file_name="cleaned_wearable.csv",
            mime="text/csv",
        )
        st.download_button(
            "Download Daily Means CSV",
            data=metrics.daily_means.reset_index().to_csv(index=False).encode("utf-8"),
            file_name="daily_means.csv",
            mime="text/csv",
        )

    except Exception as e:
        st.error(f"Failed to process file: {e}")
